# Modo residual do STRATA numa T4

Antes de rodar: *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*. Depois *Executar tudo*.

O estrato 0 é o Q4_K_M do `llama-quantize`. Cada nível seguinte divide a célula da grade do Q4_K ao meio (custa perto de 1 bit/peso) e lê o centro da sub-célula. Mede o ΔPPL no wikitext-2 de teste inteiro de: BF16, Q4_K_M (nível 0), níveis 1, 2 e 3, e na mesma rodada os GGUFs Q5_K_M e Q6_K para comparar. A pergunta: o nível 1 (~5,9 bits) chega perto do Q5_K_M e o nível 2 (~6,9) do Q6_K? Leva uns 40 minutos.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRANCH = "claude/project-thread-qjksmf"
!rm -rf strata && git clone -q --depth 1 -b $BRANCH https://github.com/Cantyll/strata.git
!rm -rf llama.cpp && git clone -q --depth 1 https://github.com/ggml-org/llama.cpp.git
!pip install -q -U transformers datasets safetensors
!pip install -q -e llama.cpp/gguf-py
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF > /dev/null && cmake --build build -j --target llama-quantize > /dev/null && ls build/bin

Converter para GGUF BF16 e quantizar, sem imatrix (o `llama-quantize` padrão).

In [ ]:
from huggingface_hub import snapshot_download
src = snapshot_download("unsloth/Llama-3.2-1B", allow_patterns=["*.json", "*.safetensors", "tokenizer*"])
!mkdir -p /content/gguf && python llama.cpp/convert_hf_to_gguf.py {src} --outtype bf16 --outfile /content/gguf/bf16.gguf 2>&1 | tail -1
for q in ["Q4_K_M", "Q5_K_M", "Q6_K"]:
    !llama.cpp/build/bin/llama-quantize /content/gguf/bf16.gguf /content/gguf/{q}.gguf {q} > /content/gguf/{q}.log 2>&1
!ls -la /content/gguf

In [ ]:
%cd /content/strata
!python tests/ppl/strata_residual.py --ppl --model unsloth/Llama-3.2-1B --gguf Q4_K_M=/content/gguf/Q4_K_M.gguf --levels 3 --ref Q5_K_M=/content/gguf/Q5_K_M.gguf,Q6_K=/content/gguf/Q6_K.gguf --out /content/res
from IPython.display import Markdown, display
display(Markdown(open('/content/res/unsloth__Llama-3.2-1B_residual_ppl.md').read()))

Cole a tabela na thread (ou baixe os arquivos abaixo).

In [ ]:
from google.colab import files
files.download('/content/res/unsloth__Llama-3.2-1B_residual_ppl.md')
files.download('/content/res/unsloth__Llama-3.2-1B_residual_ppl.json')